# Run the SQL queries

This notebook runs the two SQL files in the `sql/` folder with **DuckDB**, a small database that runs inside Python. No database server is needed.

How to use it: click **Run All** at the top. The first run takes about a minute, because it installs packages and creates the data.

In [1]:
# Step 1: install the packages this project needs (only slow the first time)
%pip install -q -r requirements.txt

In [2]:
# Step 2: make sure the (synthetic) transaction data exists
from pathlib import Path
import generate_data                      # our own file: generate_data.py

data_file = generate_data.OUT_FILE        # = data/transactions.csv
if not data_file.exists():                # first run: create it (about 1 minute)
    data_file.parent.mkdir(exist_ok=True)
    generate_data.generate().to_csv(data_file, index=False)
print("Data file:", data_file.relative_to(data_file.parent.parent))

Data file: data/transactions.csv


In [3]:
# Step 3: open DuckDB and load the CSV file as a table called 'transactions'
import duckdb

con = duckdb.connect()                    # an in-memory database (nothing saved to disk)
con.sql(f"""
    CREATE TABLE transactions AS
    SELECT * FROM read_csv_auto('{data_file}')
""")
# Quick check: how many rows, and what do they look like?
n_rows = con.sql("SELECT COUNT(*) AS n_rows FROM transactions").df()["n_rows"].iloc[0]
print(n_rows, "rows")
con.sql("SELECT * FROM transactions LIMIT 5").df()

358365 rows


,txn_id,timestamp,store_id,cashier_id,txn_type,payment_method,amount_sek,discount_pct,has_receipt,minutes_since_sale,is_fraud,fraud_type
0,T0000000,2025-10-01 09:00:00,S05,C05G,sale,card,52.50,0.0,NaN,NaN,0,None
1,T0000001,2025-10-01 09:00:00,S07,C07F,sale,cash,648.68,0.0,NaN,NaN,0,None
2,T0000002,2025-10-01 09:00:00,S07,C07A,sale,gift_card,211.05,0.0,NaN,NaN,0,None
3,T0000003,2025-10-01 09:00:00,S04,C04C,sale,card,91.54,0.0,NaN,NaN,0,None
4,T0000004,2025-10-01 09:00:00,S01,C01C,sale,cash,269.68,0.0,NaN,NaN,0,None


## Business questions (`sql/business_questions.sql`)
The file holds three queries. Each one starts with a line `-- name: ...`, so we split the file there and run them one by one.

In [4]:
# Step 4: read the file and split it into separate queries
text = Path("sql/business_questions.sql").read_text()
blocks = text.split("\n-- name: ")[1:]   # skip the header comment at the top

queries = {}
for block in blocks:
    lines = block.splitlines()
    name = lines[0]                       # the first line is the query's name
    queries[name] = "\n".join(lines[1:])  # the rest is the SQL itself
print("Queries found:", list(queries))

Queries found: ['monthly_payment_mix', 'refunds_without_receipt_by_store', 'top_cashier_per_store']


In [5]:
# Query 1: share of sales by payment method, per month
con.sql(queries["monthly_payment_mix"]).df()

,month,payment_method,share_pct
0,2025-10-01,card,83.2
1,2025-10-01,cash,9.6
2,2025-10-01,gift_card,2.8
3,2025-10-01,swish,4.4
4,2025-11-01,card,82.4
5,2025-11-01,cash,9.6
6,2025-11-01,gift_card,3.1
7,2025-11-01,swish,4.9
8,2025-12-01,card,82.4
9,2025-12-01,cash,9.4


In [6]:
# Query 2: refunds without a receipt, per store
con.sql(queries["refunds_without_receipt_by_store"]).df()

,store_id,refunds_no_receipt,value_sek,pct_of_refunds
0,S07,263,96893.0,21.8
1,S05,233,71846.0,19.8
2,S06,89,18213.0,14.2
3,S02,154,26408.0,13.6
4,S03,73,11873.0,13.2
5,S04,105,17759.0,11.7
6,S01,116,24641.0,10.7
7,S08,68,12259.0,10.6


In [7]:
# Query 3: the cashier with the highest exception rate in each store
con.sql(queries["top_cashier_per_store"]).df()

,store_id,cashier_id,exceptions_per_100_sales,rank_in_store
0,S08,C08E,9.574695,1
1,S06,C06B,9.572347,1
2,S07,C07G,7.613941,1
3,S05,C05D,7.157950,1
4,S04,C04E,5.757176,1
5,S02,C02D,5.596621,1
6,S01,C01E,5.300582,1
7,S03,C03F,5.264435,1


## Model features (`sql/exception_features.sql`)
One longer query: it builds the features used by the fraud model for every refund, void and discounted sale.

In [8]:
# Step 5: run the feature query and look at the first rows
features = con.sql(Path("sql/exception_features.sql").read_text()).df()
print(len(features), "exception transactions,", features.shape[1], "columns")
features.head()

18754 exception transactions, 17 columns


,txn_id,store_id,cashier_id,is_fraud,value_at_risk,hour,late_hour,is_cash,no_receipt,discount_pct,round_amount,is_refund,is_void,minutes_since_sale,log_amount,cashier_exception_rate_z,cashier_late_share_z
0,T0000048,S07,C07D,0,87.890,10.716667,0,0,0,0.25,0,0,0,0.0,4.487400,-0.289206,-0.764735
1,T0000051,S06,C06D,0,20.324,10.766667,0,0,0,0.20,0,0,0,0.0,3.059833,-0.042257,0.165801
2,T0000096,S02,C02E,0,50.646,11.366667,0,0,0,0.30,0,0,0,0.0,3.944413,0.992192,1.282039
3,T0000123,S04,C04F,0,33.558,11.650000,0,0,0,0.20,0,0,0,0.0,3.542639,0.488482,-1.669542
4,T0000240,S07,C07B,1,208.570,12.750000,0,0,1,0.00,0,1,0,0.0,5.345058,1.442395,1.785024


## Try your own SQL
Change the query below and run the cell again.

In [9]:
# Example: average refund amount per payment method
con.sql("""
    SELECT payment_method,
           COUNT(*)                  AS refunds,
           ROUND(AVG(amount_sek), 0) AS avg_refund_sek
    FROM transactions
    WHERE txn_type = 'refund'
    GROUP BY payment_method
    ORDER BY refunds DESC
""").df()

,payment_method,refunds,avg_refund_sek
0,card,5856,188.0
1,cash,810,284.0
2,swish,463,192.0
3,gift_card,200,199.0
